In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from scipy.stats import pearsonr

from scAnalysis import (
    sc_io,
    preprocessing,
    quality_control,
    cell_cycle,
    batch_correction,
    dimensionality,
    clustering,
    trajectory,
    differential,
    enrichment,
    visualization,
    interactive_viz,
    imputation,
)

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
local_h5ad_path = "resources/grn_benchmark/inference_data/nakatake_rna.h5ad"
tf_list_path = "resources/grn_benchmark/prior/tf_all.csv"

data = sc_io.read_h5ad(local_h5ad_path)
data.var.index = sc_io._make_unique(data.var.index.values)
print(f"Loaded: {data.n_obs} samples (bulk) and {data.n_vars} genes.")

IO: Reading H5AD from 'resources/grn_benchmark/inference_data/nakatake_rna.h5ad' ...
IO: Loaded 460 cells × 25,090 genes.
Loaded: 460 samples (bulk) and 25090 genes.


In [3]:
data = preprocessing.filter_genes(data, min_cells=3)


preprocessing.normalize_total(data, target_sum=1e4)
preprocessing.log1p(data)


tf_all = pd.read_csv(tf_list_path, header=None)[0].tolist()
available_tfs = [tf for tf in tf_all if tf in data.var.index]
all_genes = data.var.index.tolist()
print(f"Found {len(available_tfs)} Transcription Factors in the dataset.")

filter_genes: keeping 25,090 / 25,090 genes.
Found 1876 Transcription Factors in the dataset.


In [4]:
X_matrix = data.X.toarray() if sp.issparse(data.X) else data.X
n_samples = X_matrix.shape[0]

X_mean = X_matrix.mean(axis=0)
X_std = X_matrix.std(axis=0)
X_std[X_std == 0] = 1e-12

Z_matrix = (X_matrix - X_mean) / X_std

# Extract TF profiles
tf_indices = [all_genes.index(tf) for tf in available_tfs]
Z_tf = Z_matrix[:, tf_indices]

# Compute Pearson correlation matrix
corr_matrix = np.dot(Z_tf.T, Z_matrix) / n_samples
weight_matrix = np.abs(corr_matrix)

for i, tf_idx in enumerate(tf_indices):
    weight_matrix[i, tf_idx] = 0.0


flat_weights = weight_matrix.flatten()
top_k = min(50000, len(flat_weights))

top_indices = np.argpartition(flat_weights, -top_k)[-top_k:]
top_indices = top_indices[np.argsort(flat_weights[top_indices])[::-1]]

tf_idx_2d, gene_idx_2d = np.unravel_index(top_indices, weight_matrix.shape)


edges = []
for i in range(len(top_indices)):
    weight = weight_matrix[tf_idx_2d[i], gene_idx_2d[i]]
    if weight > 0.05:
        edges.append({
            'source': available_tfs[tf_idx_2d[i]],
            'target': all_genes[gene_idx_2d[i]],
            'weight': str(weight) # Weight must be a string for geneRNIB
        })

grn_df = pd.DataFrame(edges)


output_anndata = ad.AnnData(
    X=np.empty((0, 0)),
    uns={
        "method_id": "scAnalyzer_Pearson",
        "dataset_id": "nakatake",
        "prediction": grn_df[["source", "target", "weight"]]
    }
)

os.makedirs("output", exist_ok=True)
output_path = "output/nakatake_scAnalyzer_GRN.h5ad"
output_anndata.write_h5ad(output_path)

print(f"Total number of edges extracted: {len(grn_df)}")

Total number of edges extracted: 50000


In [9]:
%cd task_grn_inference

/home/ayyuce/Desktop/task_grn_inference


/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [21]:
!pwd

/home/ayyuce/Desktop/task_grn_inference


In [13]:
#!rm resources/resources

In [14]:
!ln -s ../resources resources
!ls -l resources/grn_benchmark/inference_data/nakatake_rna.h5ad

-rw-rw-r-- 1 ayyuce ayyuce 283939072 Nov 14  2025 resources/grn_benchmark/inference_data/nakatake_rna.h5ad


In [16]:
!bash scripts/prior/run_consensus.sh \
  --dataset nakatake \
  --new_model ../output/nakatake_scAnalyzer_GRN.h5ad

Config file generated at: src/utils/config.env
Adding new model: ../output/nakatake_scAnalyzer_GRN.h5ad
../output/nakatake_scAnalyzer_GRN.h5ad
Running consensus for Regression
Running regression consensus for dataset: nakatake
{'dataset': 'nakatake', 'evaluation_data': 'resources/grn_benchmark/inference_data/nakatake_rna.h5ad', 'regulators_consensus': 'resources/grn_benchmark/prior/regulators_consensus_nakatake.json', 'predictions': ['../output/nakatake_scAnalyzer_GRN.h5ad']}
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_links: (50000, 3)
Sparsity of ../output/nakatake_scAnalyzer_GRN.h5ad: 0.999920572904463
Running consensus for ws distance
Skipping dataset: nakatake


In [17]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       3.1Gi       4.2Gi       421Mi       988Mi       4.5Gi
Swap:           19Gi       9.0Gi        10Gi


In [23]:
!env JOBLIB_TEMP_FOLDER=../joblib_tmp MPLBACKEND=agg bash src/metrics/all_metrics/run_local.sh \
  --dataset nakatake \
  --prediction ../output/nakatake_scAnalyzer_GRN.h5ad \
  --score ../output/nakatake_score.h5ad \
  --num_workers 4

Layer is set to: lognorm
Regression type is set to: ridge
Number of workers is set to: 4
Dataset is set to: nakatake
Prediction file is set to: ../output/nakatake_scAnalyzer_GRN.h5ad
Score file is set to: ../output/nakatake_score.h5ad
Method id: scAnalyzer_Pearson, Dataset id: nakatake
Computing metrics for dataset nakatake: ['regression', 'gs_recovery', 'vc']
Computing metric: regression
Layer lognorm not found, using X_norm instead
Evaluating 25090 genes (consensus data available for all)
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_links: (50000, 3)
Static approach (theta=r_precision):
Static approach (theta=r_recall):
Raw approach (no theta):
theta    r2_raw  r_precision  r_recall
0      0.223923     0.189179  0.189179
Computing metric: gs_recovery
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_li

In [24]:
import anndata as ad
import pandas as pd
from IPython.display import display

score_data = ad.read_h5ad("../output/nakatake_score.h5ad")

metric_ids = score_data.uns['metric_ids']
metric_values = score_data.uns['metric_values']

df_scores = pd.DataFrame({
    'Metric': metric_ids,
    'Score (scAnalyzer)': metric_values
})

#df_scores['Score (scAnalyzer)'] = df_scores['Score (scAnalyzer)'].round(3)
df_scores = df_scores.sort_values(by='Score (scAnalyzer)', ascending=False).reset_index(drop=True)

display(df_scores)
df_scores.to_csv("../output/scAnalyzer_nakatake_results.csv", index=False)

,Metric,Score (scAnalyzer)
0,bioplanet_2019_gs_n_active,973.0
1,hallmark_2020_gs_n_active,41.0
2,wikipathways_2019_gs_n_active,310.0
3,go_bp_2023_gs_n_active,2487.0
4,kegg_2021_gs_n_active,230.0
5,reactome_2022_gs_n_active,1143.0
6,reactome_2022_gs_precision,0.9291044776119403
7,hallmark_2020_gs_precision,0.8666666666666667
8,go_bp_2023_gs_precision,0.864406779661017
9,wikipathways_2019_gs_precision,0.8541666666666666
